# 2. Structural actuation limits

Three gain-independent limits on what the actuator can do. Torque is
perpendicular to the pod's dipole moment (`tau = m x B`), so a single-point-dipole
pod has no body-frame yaw authority; the multi-dipole model recovers full
6-of-6 rank through the same coils; and Classic Track's per-row Jacobian is
rank-deficient in the attitude block. Guards in `tests/test_control.py`.

In [1]:
from nblib.bootstrap import REPO_ROOT
from nblib import actuator_jacobian as aj

import numpy as np
import matplotlib.pyplot as plt

FIG_DIR = REPO_ROOT / "docs" / "results_figures"
DOF = aj.DOF_LABELS

## 2a. Torque is perpendicular to the dipole moment

Fix `m` vertical, sweep `B` over the unit sphere, plot `tau = m x B`. Every
result lies in the horizontal plane; no tilt of `B` gives a vertical (yaw)
component.

In [ ]:
m = np.array([0.0, 0.0, 1.0])
azimuths = np.linspace(0, 2 * np.pi, 18, endpoint=False)
elevations = np.linspace(-np.pi / 2 * 0.85, np.pi / 2 * 0.85, 7)

B_dirs = []
taus = []
for el in elevations:
    for az in azimuths:
        B = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
        B_dirs.append(B)
        taus.append(np.cross(m, B))

B_dirs = np.array(B_dirs)
taus = np.array(taus)

max_tz = np.max(np.abs(taus[:, 2]))
print(f"max |tau_z| across {len(taus)} field directions = {max_tz:.2e}  (exactly zero by construction)")

In [ ]:
fig = plt.figure(figsize=(12, 5.5))

# Left: the torque vectors in 3D.
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
for tau in taus:
    ax1.quiver(0, 0, 0, *tau, color="blue", alpha=0.5,
               linewidth=1.3, arrow_length_ratio=0.15)
ax1.quiver(0, 0, 0, 0, 0, 1.05, color="black", linewidth=3.5, arrow_length_ratio=0.12)
ax1.text(0, 0, 1.15, "m (fixed vertical)", fontsize=10, ha="center", fontweight="bold")

xx, yy = np.meshgrid(np.linspace(-1, 1, 2), np.linspace(-1, 1, 2))
ax1.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.12, color="grey")

ax1.set_xlim(-1, 1)
ax1.set_ylim(-1, 1)
ax1.set_zlim(-0.2, 1.2)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")
ax1.set_title(f"tau = m x B  ({len(taus)} field directions)", fontsize=10.5)

# Right: the same torques seen edge-on (in-plane magnitude vs vertical component).
ax2 = fig.add_subplot(1, 2, 2)
for tau in taus:
    ax2.plot([0, np.hypot(tau[0], tau[1])], [0, tau[2]], color="blue", alpha=0.4, lw=1.2)
ax2.axhline(0, color="black", lw=2, label="tau_z")
ax2.set_xlabel("|tau| in-plane component")
ax2.set_ylabel("tau_z (vertical)")
ax2.set_ylim(-0.5, 0.5)
ax2.set_title("Side view", fontsize=10.5)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(alpha=0.3)

fig.suptitle("Torque is perpendicular to m", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIG_DIR / "fig7_torque_perpendicular_field.png", dpi=150)
plt.show()

Under the point-dipole model, body-frame yaw (torque along `m`) is exactly zero
for any coil current pattern. It is a rotation identity, not a tuning failure.

## 2b. Full 6-DOF authority with the multi-dipole model

One unit wrench per DOF through the `CurrentAllocator`, level and on-centre. This
is the pose the point model calls unreachable for `Tz`. Bar height is current
cost (% of `current_limit`).

In [4]:
scene = aj.load_scene()
ratios, currents, realized = aj.six_dof_authority(scene)
pct_current = 100 * currents / scene.current_limit
for label, r, p in zip(DOF, ratios, pct_current):
    print(f"{label}: realized/target={r:.4f}   current used={p:.2f}% of limit")

Fx: realized/target=1.0000   current used=2.30% of limit
Fy: realized/target=1.0000   current used=0.99% of limit
Fz: realized/target=1.0000   current used=0.62% of limit
Tx: realized/target=1.0000   current used=2.38% of limit
Ty: realized/target=1.0000   current used=14.54% of limit
Tz: realized/target=1.0000   current used=2.46% of limit


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))

# Blue = forces (Fx, Fy, Fz), red = torques (Tx, Ty, Tz).
bars = ax.bar(DOF, pct_current, color=["blue"] * 3 + ["red"] * 3)
for bar, r in zip(bars, ratios):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + pct_current.max() * 0.02,
            f"{100 * r:.0f}% realized", ha="center", fontsize=9)

ax.set_ylim(0, pct_current.max() * 1.2)
ax.set_ylabel("current (% of limit)")
ax.set_title("Current cost per unit wrench, by DOF")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_six_dof_authority.png", dpi=150)
plt.show()

Full 6/6 rank recovered through the same coil array, no new hardware. The cost
to realise each axis varies by about 24x.

## 2c. Where yaw is rank-deficient

The 6x6 realised-vs-commanded matrix under `pod_model="point"` at three poses,
with `Tz` targeted like `Tx`/`Ty`. Level: the `Tz` row and column are a
structural zero. Tilted 0.2 rad: a world-frame `Tz` command projects onto the
body axes, realising ~4% on-axis and ~19% into pitch/roll. The multi-dipole
Jacobian removes this coupling.

In [6]:
POSES = {"level": (0.0, 0.0, 0.0), "roll 0.2 rad": (0.2, 0.0, 0.0), "pitch 0.2 rad": (0.0, 0.2, 0.0)}
pm_mats = {}
for name, rpy in POSES.items():
    M_pm, di_max, sv = aj.point_cross_coupling(scene, rpy)
    pm_mats[name] = M_pm
    print(f"{name}: min singular value of J6 = {sv[-1]:.2e}   "
          f"Tz-row on-diag = {M_pm[5, 5]:.3f}   Tz-row max |off-diag| = {np.abs(np.delete(M_pm[5], 5)).max():.3f}")

level: min singular value of J6 = 0.00e+00   Tz-row on-diag = 0.000   Tz-row max |off-diag| = 0.000
roll 0.2 rad: min singular value of J6 = 2.02e-19   Tz-row on-diag = 0.039   Tz-row max |off-diag| = 0.195
pitch 0.2 rad: min singular value of J6 = 7.20e-19   Tz-row on-diag = 0.039   Tz-row max |off-diag| = 0.195


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.4))
vmax = max(np.max(np.abs(m)) for m in pm_mats.values())

for ax, (name, M_pm) in zip(axes, pm_mats.items()):
    im = ax.imshow(M_pm, cmap="RdBu_r", vmin=-vmax, vmax=vmax)

    ax.set_xticks(range(6))
    ax.set_xticklabels(DOF)
    ax.set_yticks(range(6))
    ax.set_yticklabels(DOF)
    ax.set_xlabel("realised DOF")

    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{M_pm[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if abs(M_pm[i, j]) > 0.5 * vmax else "black")

    ax.set_title(name)

axes[0].set_ylabel("commanded DOF")
fig.colorbar(im, ax=axes, label="realised / commanded", fraction=0.025, pad=0.02)
fig.suptitle("Realised vs commanded wrench (point model)", fontsize=12)
fig.savefig(FIG_DIR / "fig_dof_cross_coupling_pointmodel.png", dpi=150, bbox_inches="tight")
plt.show()

## 2d. Classic Track roll/pitch ceiling (printed, no figure)

The stacked 6-target effectiveness Jacobian under per-row grouping has only 2 of
4 row groups well-conditioned. This is a ceiling on roll/pitch authority set by
the geometry, not finite-difference noise in the Jacobian.

In [8]:
S_classic, rank_classic, n_groups = aj.row_jacobian_svd()
print(f"Classic Track ({n_groups} row groups): singular values = "
      f"{np.array2string(S_classic, precision=5)}   rank = {rank_classic}/6")
print(f"  weakest well-conditioned / weakest: {S_classic[1] / S_classic[-1]:.0f}x gap")

Classic Track (4 row groups): singular values = [1.45192 1.03096 0.00505 0.00385]   rank = 4/6
  weakest well-conditioned / weakest: 268x gap
